# 09 - Hyperparameter Sweep and Multi-Seed ProtocolDr Hunter, 20 August: *"you need to do some playing around with the parameters"* and*"don't throw these away... keep a note of them, because if you don't get improvement,you can refer back to these ones."*This notebook does both. Every run appends to `writing/sweep_results.csv`. Nothing is everoverwritten, including the failures - a failed configuration is evidence about the searchspace, not waste.**Structure, so a Colab disconnect never costs more than one run:**- Section 3: reusable `train_and_eval()`- Section 4: **Stage 1** - learning-rate sweep at a fixed seed- Section 5: **Stage 2** - best LR across three seeds, reported as mean +/- std- Section 6: **Stage 3** - the same protocol for MuRIL and IndicBERT- Section 7: the full results table and the model comparison**Runtime > Change runtime type > T4 GPU.** About 5 minutes per run.

### 1. Setup

In [ ]:
try:    from google.colab import drive; drive.mount('/content/drive')except Exception as e:    print('Not on Colab / already mounted:', e)import numpy as np, pandas as pd, torch, time, random, gcfrom pathlib import Pathfrom torch.utils.data import Dataset, DataLoaderfrom transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmupfrom sklearn.metrics import f1_score, accuracy_score, confusion_matrix, precision_recall_fscore_supportDEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'print('device:', DEVICE)if DEVICE=='cpu': print('WARNING: no GPU. Runtime > Change runtime type > T4 GPU.')DATA_ROOT = Path('/content/drive/MyDrive/dissertation/data')W = Path('/content/drive/MyDrive/dissertation/writing'); W.mkdir(parents=True, exist_ok=True)FIG = W/'figures'; FIG.mkdir(exist_ok=True)LOG = W/'sweep_results.csv'

### 2. Data (loaded once, reused by every run)

In [ ]:
import sys; sys.path.insert(0, '/content/drive/MyDrive/dissertation/notebooks')from hinglish_hate import load_hasoc2022_threads, filter_romanisedtrain_df = pd.read_parquet(DATA_ROOT/'splits'/'bohra_train.parquet')test_df  = pd.read_parquet(DATA_ROOT/'splits'/'bohra_test.parquet')assert len(train_df)==3659 and len(test_df)==915, 'split changed - comparison invalid'h22_r = filter_romanised(load_hasoc2022_threads(DATA_ROOT), include_mixed=True)print(f'train {len(train_df)} | test {len(test_df)} | cross HASOC22 {len(h22_r)}')class PostDataset(Dataset):    def __init__(self, texts, labels, tok, max_len):        self.texts=list(texts); self.labels=list(labels); self.tok=tok; self.max_len=max_len    def __len__(self): return len(self.texts)    def __getitem__(self, i):        e = self.tok(str(self.texts[i]), truncation=True, padding='max_length',                     max_length=self.max_len, return_tensors='pt')        return {'input_ids':e['input_ids'].squeeze(0),                'attention_mask':e['attention_mask'].squeeze(0),                'labels':torch.tensor(self.labels[i], dtype=torch.long)}

### 3. One run, end to end`train_and_eval` is the whole experiment as a function. Every hyperparameter is an argument,so a sweep is a loop rather than repeated hand-editing - which is what caused the threeinconsistent runs on 19 August.**Every run logs `final_train_loss` and `degenerate`.** A run whose loss stays near ln(2) =0.693, or which predicts one class for more than 90% of the test set, is flagged rather thansilently reported as a low score. That distinction matters: a degenerate run says nothingabout the model's capability, only about the optimiser failing to escape a trivial solution.

In [ ]:
def set_all_seeds(s):    random.seed(s); np.random.seed(s); torch.manual_seed(s)    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)def train_and_eval(model_name, lr=2e-5, epochs=3, seed=42, batch_size=16,                   max_len=128, warmup_frac=0.1, note='', verbose=True):    set_all_seeds(seed)    tok = AutoTokenizer.from_pretrained(model_name)    tr_dl = DataLoader(PostDataset(train_df['text'], train_df['label'], tok, max_len),                       batch_size=batch_size, shuffle=True)    te_dl = DataLoader(PostDataset(test_df['text'], test_df['label'], tok, max_len),                       batch_size=batch_size*2, shuffle=False)    cr_dl = DataLoader(PostDataset(h22_r['text'], h22_r['label'], tok, max_len),                       batch_size=batch_size*2, shuffle=False)    counts  = np.bincount(train_df['label'].values)    weights = torch.tensor(len(train_df)/(2.0*counts), dtype=torch.float).to(DEVICE)    set_all_seeds(seed)   # again, right before the randomly-initialised head is created    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2).to(DEVICE)    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)    total = len(tr_dl)*epochs    sch = get_linear_schedule_with_warmup(opt, int(warmup_frac*total), total)    lossf = torch.nn.CrossEntropyLoss(weight=weights)    hist, t0 = [], time.time()    for ep in range(1, epochs+1):        model.train(); run = 0.0        for b in tr_dl:            logits = model(input_ids=b['input_ids'].to(DEVICE),                           attention_mask=b['attention_mask'].to(DEVICE)).logits            loss = lossf(logits, b['labels'].to(DEVICE)); loss.backward()            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)            opt.step(); sch.step(); opt.zero_grad(); run += loss.item()        hist.append(run/len(tr_dl))        if verbose: print(f'    epoch {ep}/{epochs} loss {hist[-1]:.4f}')    def predict(dl):        model.eval(); P, T = [], []        with torch.no_grad():            for b in dl:                lg = model(input_ids=b['input_ids'].to(DEVICE),                           attention_mask=b['attention_mask'].to(DEVICE)).logits                P.extend(lg.argmax(1).cpu().numpy()); T.extend(b['labels'].numpy())        return np.array(T), np.array(P)    yt, yp   = predict(te_dl)    yct, ycp = predict(cr_dl)    p, r, f, _ = precision_recall_fscore_support(yt, yp, zero_division=0)    frac_one  = max((yp==0).mean(), (yp==1).mean())    degenerate = bool(frac_one > 0.90 or hist[-1] > 0.68)    row = {'model':model_name, 'lr':lr, 'epochs':epochs, 'seed':seed,           'batch_size':batch_size, 'max_len':max_len,           'macro_f1':f1_score(yt,yp,average='macro'),           'hate_f1':f1_score(yt,yp,pos_label=1,zero_division=0),           'f1_not':f[0], 'accuracy':accuracy_score(yt,yp),           'precision_hate':p[1], 'recall_hate':r[1],           'cross_macro_f1':f1_score(yct,ycp,average='macro'),           'cross_hate_f1':f1_score(yct,ycp,pos_label=1,zero_division=0),           'final_train_loss':hist[-1], 'first_train_loss':hist[0],           'degenerate':degenerate, 'majority_pred_frac':round(frac_one,3),           'runtime_s':round(time.time()-t0), 'note':note,           'timestamp':pd.Timestamp.now().isoformat(timespec='seconds')}    prev = pd.read_csv(LOG) if LOG.exists() else pd.DataFrame()    pd.concat([prev, pd.DataFrame([row])], ignore_index=True).to_csv(LOG, index=False)    flag = '  [DEGENERATE]' if degenerate else ''    print(f'  -> macro-F1 {row["macro_f1"]:.3f} | hate-F1 {row["hate_f1"]:.3f} | '          f'cross {row["cross_macro_f1"]:.3f} | loss {hist[-1]:.3f}{flag}')    del model; gc.collect(); torch.cuda.empty_cache()    return rowprint('ready. chance-level loss = ln(2) =', round(float(np.log(2)),4))

### 4. Stage 1 - learning-rate sweepOne variable at a time, seed held at 42. This is the controlled version of what went wrong on19 August, where epochs and learning rate were changed together and the effect of each couldnot be separated.**Three runs, roughly 15 minutes.**

In [ ]:
MODEL = 'xlm-roberta-base'for lr in [1e-5, 2e-5, 3e-5]:    print(f'\n=== {MODEL} | lr={lr} | 3 epochs | seed 42 ===')    train_and_eval(MODEL, lr=lr, epochs=3, seed=42, note='stage1 lr sweep')

In [ ]:
sweep = pd.read_csv(LOG)s1 = sweep[sweep['note']=='stage1 lr sweep']display(s1[['lr','macro_f1','hate_f1','cross_macro_f1','final_train_loss','degenerate']].round(3))ok = s1[~s1['degenerate']]BEST_LR = float(ok.loc[ok['macro_f1'].idxmax(),'lr']) if len(ok) else 2e-5print(f'\nbest non-degenerate LR: {BEST_LR}')

### 5. Stage 2 - multi-seed at the best learning rateHunter's other instruction was to keep every result. This is why: with three seeds you canreport a mean and a standard deviation, which is what the statistical baseline already getsfrom five-fold CV. Quoting a single transformer run against a cross-validated baseline is nota fair comparison.**Three runs, roughly 15 minutes.**

In [ ]:
for seed in [42, 1337, 2024]:    print(f'\n=== {MODEL} | lr={BEST_LR} | 3 epochs | seed {seed} ===')    train_and_eval(MODEL, lr=BEST_LR, epochs=3, seed=seed, note='stage2 multiseed')

In [ ]:
sweep = pd.read_csv(LOG)s2 = sweep[(sweep['note']=='stage2 multiseed') & (sweep['model']==MODEL)]display(s2[['seed','macro_f1','hate_f1','cross_macro_f1','final_train_loss','degenerate']].round(3))print(f"\n{MODEL}, {len(s2)} seeds at lr={BEST_LR}")print(f"  within macro-F1 {s2['macro_f1'].mean():.3f} +/- {s2['macro_f1'].std(ddof=0):.3f}")print(f"  hate-F1        {s2['hate_f1'].mean():.3f} +/- {s2['hate_f1'].std(ddof=0):.3f}")print(f"  cross macro-F1 {s2['cross_macro_f1'].mean():.3f} +/- {s2['cross_macro_f1'].std(ddof=0):.3f}")print(f"\nbaseline for comparison: within 0.657 +/- 0.005 (5-fold) | cross 0.510")

### 6. Stage 3 - MuRIL and IndicBERTSame protocol, different pretraining. **MuRIL is the one with a specific reason to workhere:** Khanuja et al. (2021) train it on transliterated Latin-script Indian-language pairsusing the Dakshina dataset, and report results on transliterated test sets. XLM-R has no suchtargeted exposure. If romanisation is the difficulty, MuRIL should show it.**Six runs, roughly 35 minutes.** Run one model at a time if Colab is unstable.

In [ ]:
for m in ['google/muril-base-cased', 'ai4bharat/indic-bert']:    for seed in [42, 1337, 2024]:        print(f'\n=== {m} | lr={BEST_LR} | 3 epochs | seed {seed} ===')        try:            train_and_eval(m, lr=BEST_LR, epochs=3, seed=seed, note='stage3 other models')        except Exception as e:            print(f'  FAILED: {type(e).__name__}: {str(e)[:200]}')

### 7. Full resultsNothing is filtered out. Degenerate runs stay in the table, flagged - that is the recordHunter asked for.

In [ ]:
sweep = pd.read_csv(LOG)print(f'{len(sweep)} runs logged | {sweep["degenerate"].sum()} degenerate\n')display(sweep[['model','lr','epochs','seed','macro_f1','hate_f1','cross_macro_f1',               'final_train_loss','degenerate','note']].round(3))clean = sweep[~sweep['degenerate']]summary = (clean.groupby('model')           .agg(n=('macro_f1','size'),                within_mean=('macro_f1','mean'), within_std=('macro_f1','std'),                hate_mean=('hate_f1','mean'),                cross_mean=('cross_macro_f1','mean'), cross_std=('cross_macro_f1','std'))           .round(3).reset_index())print('\nNon-degenerate runs only:')display(summary)

In [ ]:
import matplotlib.pyplot as pltrows = [{'model':'TF-IDF + LogReg\n(baseline)','within':0.657,'within_err':0.005,         'cross':0.510,'cross_err':0.0}]for _, r in summary.iterrows():    rows.append({'model':r['model'].split('/')[-1],                 'within':r['within_mean'], 'within_err':(r['within_std'] or 0),                 'cross':r['cross_mean'],  'cross_err':(r['cross_std'] or 0)})plot_df = pd.DataFrame(rows)x = np.arange(len(plot_df)); w = 0.35fig, ax = plt.subplots(figsize=(9,5))ax.bar(x-w/2, plot_df['within'], w, yerr=plot_df['within_err'], capsize=4,       label='within-dataset (Bohra)', edgecolor='black')ax.bar(x+w/2, plot_df['cross'],  w, yerr=plot_df['cross_err'], capsize=4,       label='cross-dataset (-> HASOC22)', edgecolor='black')ax.set_xticks(x, plot_df['model'], rotation=12); ax.set_ylabel('macro-F1'); ax.set_ylim(0,1)ax.axhline(0.5, color='grey', ls=':', lw=1)ax.set_title('Within vs cross-dataset macro-F1, mean +/- std over seeds')ax.legend(); plt.tight_layout()plt.savefig(FIG/'model_comparison_multiseed.png', dpi=150, bbox_inches='tight'); plt.show()plot_df.to_csv(W/'model_comparison_multiseed.csv', index=False)print('saved model_comparison_multiseed.png / .csv')

### Notes for the write-up- Every configuration tried is in `sweep_results.csv`, degenerate runs included. The negative  results are part of the record, per Hunter's instruction on 20 August.- Fine-tuning instability in BERT-family models on small datasets is documented: Dodge et al.  (2020) and Mosbach et al. (2021). Cite both when reporting the variance.- Report transformers as mean +/- std over seeds, matching how the baseline is reported.- **Still to build:** whole-post romanisation attack (Objective 4), five-fold CV on the best  transformer, and the LoRA LLM if scope allows.